In [20]:
import struct
import numpy as np

In [21]:
def load_mnist_images(filename): # Loading images
    with open(filename, 'rb') as f:
        _, num, rows, cols = struct.unpack(">IIII", f.read(16))
        images = np.fromfile(f, dtype=np.uint8).reshape(num, rows * cols)
    return images / 255.0  # Normalize pixel values to [0, 1]


def load_mnist_labels(filename): # Loading labels
    with open(filename, 'rb') as f:
        _, num = struct.unpack(">II", f.read(8))
        labels = np.fromfile(f, dtype=np.uint8)
    return labels


def one_hot_encode(labels, num_classes=10): # Puts labels in proper format
    return np.eye(num_classes)[labels]

In [22]:
y0_train = load_mnist_images('train-images-idx3-ubyte')
t_train = one_hot_encode(load_mnist_labels('train-labels-idx1-ubyte'))
y0_test = load_mnist_images('t10k-images-idx3-ubyte')
t_test = one_hot_encode(load_mnist_labels('t10k-labels-idx1-ubyte'))

In [41]:
def initialize_weights(input_size, hidden_size, output_size):
    W0 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
    W1 = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / hidden_size)
    return W0, W1

def forward_pass(y0, W0, W1):
    z0 = np.dot(W0, y0)
    y1 = np.tanh(z0)
    z1 = np.dot(W1, y1)
    y2 = np.tanh(z1)
    return z0, y1, z1, y2

def compute_loss(y2, t):
    loss = 0.5 * np.sum(np.sqrt((y2-t)**2))**2
    return loss

def backpropagation(y0, t, z0, y1, z1, y2, W0, W1, learning_rate):
    delta1 = - (y2 - t) * (1. - y2**2)
    delta0 = np.dot(delta1.T, W1) * (1. - y1**2)

    W1 += learning_rate * np.outer(delta1, y1)
    W0 += learning_rate * np.outer(delta0, y0)

    return W0, W1

def train_neural_network(y0_train, t_train, y0_test, t_test, epochs, learning_rate):
    input_size = 784
    hidden_size = 784
    output_size = 10
    
    W0, W1 = initialize_weights(input_size, hidden_size, output_size)
    
    for epoch in range(epochs):

        for i, y0 in enumerate(y0_train):
            
            z1, a1, z2, a2 = forward_pass(y0, W0, W1)

            # Compute loss
            loss = compute_loss(t_train[i], a2)

            # Backpropagation
            W0, W1 = backpropagation(y0, t_train[i], z1, a1, z2, a2, W0, W1, learning_rate)
        
        print(f'Epoch {epoch}, Loss: {loss}')

    # Evaluate on test data
    _, _, _, test_predictions = forward_pass(y0_test, W0, W1)
    test_accuracy = np.mean(np.argmax(test_predictions, axis=1) == np.argmax(t_test, axis=1))
    print(f'Test accuracy: {test_accuracy * 100:.2f}%')


In [42]:
train_neural_network(y0_train, t_train, y0_test, t_test, epochs=10, learning_rate=0.01)

Epoch 0, Loss: 0.618095185028427
Epoch 1, Loss: 0.4569347099059803
Epoch 2, Loss: 0.3140778800345338
Epoch 3, Loss: 0.3229086440510489
Epoch 4, Loss: 0.3142101958254716
Epoch 5, Loss: 0.2944492636288391
Epoch 6, Loss: 0.26909658297354533
Epoch 7, Loss: 0.24346725275640357
Epoch 8, Loss: 0.22109264410218715
Epoch 9, Loss: 0.2043028379776386


ValueError: shapes (784,784) and (10000,784) not aligned: 784 (dim 1) != 10000 (dim 0)

In [5]:
W0, W1 = initialize_weights(784, 784, 10)

In [19]:
print(np.shape(W0))
print(np.shape(W1))
print(np.dot(W1, np.random.rand(784)))

(784, 784)
(10, 784)
[-0.74643611 -0.97402995 -0.74615385 -0.73351744  0.02541748 -0.29999025
 -0.57315014 -0.51739554 -0.5081864  -0.54092365]
